In [1]:
from skimage import io
import numpy as np
from glob import glob
import sys
sys.path.append('../')
sys.path.append('/home/philipp/PycharmProjects/ps_course_projects-clb2_data_processing/src/')

from tracking import track_position
from napari_rf.features import FeatureCreator
from napari_rf.RF import RF
from joblib import load
from pathlib import Path
import re
import os
import matplotlib.pyplot as plt
from skimage.morphology import binary_erosion, binary_dilation
from tqdm import tqdm
import scipy.ndimage as ndi
import warnings
warnings.filterwarnings('ignore')

In [2]:
feature_creator = FeatureCreator()

In [3]:
def create_composite_features(feature_creator, *imgs):
    features = []
    for img in imgs:
        features.append(feature_creator.make_simple_features(img))
        
    return np.concatenate(features, axis=-1)

In [4]:
mappings = {
#     '/media/philipp/WD1/clb2/processed_data/20230511_CLB2cellcycle_fovs002/':
#     {
# #         'wt': [5,6,7,8,9],
# #         'zip': [17,18,19], #15, !!!16
#         'she2': [21,22,23,24], # 20,
# #         'she3': [10,11,12,13,14],
#     },
    '/media/philipp/WD1/clb2/processed_data/20230511_CLB2cellcycle_fovs003/':
    {
#         'wt': [10,11,12,13,14],
        'zip': [17,18,19],# 15,16,
#         'she2': [5,6,7,8,9], # CONTINUE HERE
#         'she3': [20,21,22,23,24],
    },
    '/media/philipp/WD1/clb2/processed_data/20230512_CLB2cellcycle_fovs004/':
    {
        'wt': [5,6,7,8,9],
#         'zip': [15,16,17,18,19],
#         'she2': [20,21,22,23,24], # CONTINUE HERE
#         'she3': [10,11,12,13,14],
    },
}

In [5]:
classifiers = {
    'wt': {
        'neck_clf' : load('/media/philipp/WD1/clb2/classifiers/necks.joblib'),
        'spindle_clf' : load('/media/philipp/WD1/clb2/classifiers/spindle_nuclei.joblib')
        },
    'she2': {
        'neck_clf' : load('/media/philipp/WD1/clb2/classifiers/necks.joblib'),
        'spindle_clf' : load('/media/philipp/WD1/clb2/classifiers/spindle_nuclei.joblib')
        },
    'she3': {
        'neck_clf' : load('/media/philipp/WD1/clb2/classifiers/necks.joblib'),
        'spindle_clf' : load('/media/philipp/WD1/clb2/classifiers/spindle_nuclei.joblib')
        },
    'zip': {
        'neck_clf' : load('/media/philipp/WD1/clb2/classifiers/necks_zip_mutant.joblib'),
        'spindle_clf' : load('/media/philipp/WD1/clb2/classifiers/spindle_nuclei_zip_mutant.joblib')
        }
}

In [ ]:
for experiment, strains in mappings.items():
    print(experiment)
    for strain, positions in strains.items():
        for position in positions:
            path = f"{experiment}/position_{str(position).zfill(2)}/z_level_0/"
            os.makedirs(f"{path}/active_neck_proba", exist_ok=True)
            os.makedirs(f"{path}/inactive_neck_proba", exist_ok=True)
            os.makedirs(f"{path}/spindle_proba", exist_ok=True)
            os.makedirs(f"{path}/neck_segmentation", exist_ok=True)
            os.makedirs(f"{path}/neck_tracks", exist_ok=True)
            
            segmentation_imgs = sorted(glob(f"{path}/tracks_0/*"))
            
            for img_path in tqdm(segmentation_imgs):
                try:
                    img_name = f'{Path(img_path).stem}.tif'
                    channel_1 = io.imread(f"{path}/background_corrected_1/{img_name}")
                    channel_2 = io.imread(f"{path}/background_corrected_2/{img_name}")

                    features = create_composite_features(feature_creator, channel_1, channel_2)

                    neck_predictions = classifiers[strain]['neck_clf'].predict_segmenter(features)
                    spindle_predictions = classifiers[strain]['spindle_clf'].predict_segmenter(features)

                    io.imsave(f'{path}/active_neck_proba/{img_name}', neck_predictions[3].astype(np.float16))
                    io.imsave(f'{path}/inactive_neck_proba/{img_name}', neck_predictions[4].astype(np.float16))
                    io.imsave(f'{path}/spindle_proba/{img_name}', spindle_predictions[3].astype(np.float16))


                    q = np.argmax(neck_predictions, axis=0) == 3

                    for a in range(4):
                        q = binary_erosion(q)
                    for a in range(4):
                        q = binary_dilation(q)
                    labels, _ = ndi.label(q)

                    io.imsave(f"{path}/neck_segmentation/{Path(img_path).stem}.tif", labels.astype(np.uint16))
                except:
                    print(f'!FAILED {img_path}')

            track_position(f"{path}/neck_segmentation", f"{path}/neck_tracks")

/media/philipp/WD1/clb2/processed_data/20230511_CLB2cellcycle_fovs003/


100%|███████████████████████████████████████████| 73/73 [25:38<00:00, 21.07s/it]
73it [01:18,  1.08s/it]
72it [00:00, 3469.51it/s]
72it [00:00, 122.73it/s]
100%|███████████████████████████████████████████| 73/73 [43:09<00:00, 35.47s/it]
73it [01:28,  1.21s/it]
72it [00:01, 55.38it/s]
72it [00:01, 38.20it/s] 
100%|███████████████████████████████████████████| 73/73 [30:47<00:00, 25.31s/it]
73it [01:29,  1.23s/it]
72it [00:00, 1817.30it/s]
72it [00:01, 46.60it/s] 
100%|███████████████████████████████████████████| 73/73 [00:46<00:00,  1.58it/s]


/media/philipp/WD1/clb2/processed_data/20230512_CLB2cellcycle_fovs004/


100%|███████████████████████████████████████████| 73/73 [30:30<00:00, 25.08s/it]
73it [00:49,  1.47it/s]
72it [00:00, 3862.06it/s]
72it [00:00, 207.63it/s]
100%|███████████████████████████████████████████| 73/73 [32:57<00:00, 27.09s/it]
73it [01:28,  1.21s/it]
72it [00:00, 2262.91it/s]
72it [00:00, 89.63it/s] 
100%|███████████████████████████████████████████| 73/73 [33:18<00:00, 27.37s/it]
73it [01:28,  1.21s/it]
72it [00:00, 1766.05it/s]
72it [00:01, 47.38it/s] 
100%|███████████████████████████████████████████| 73/73 [38:24<00:00, 31.56s/it]
73it [00:55,  1.32it/s]
72it [00:00, 1927.70it/s]
72it [00:01, 67.33it/s] 
100%|███████████████████████████████████████████| 73/73 [25:31<00:00, 20.98s/it]
73it [00:20,  3.54it/s]
72it [00:00, 2068.44it/s]
72it [00:01, 70.61it/s] 
 67%|████████████████████████████▊              | 49/73 [00:07<00:04,  5.09it/s]

In [ ]:
experiment, position

In [ ]:
for path in glob(f'{root}/*'):
    position = re.findall('position_(\d+)', path)
    os.makedirs(f"{path}/active_neck_proba", exist_ok=True)
    os.makedirs(f"{path}/inactive_neck_proba", exist_ok=True)
    os.makedirs(f"{path}/spindle_proba", exist_ok=True)
    if position:
        strain = pos_to_strain.get(int(position[0]))
        
        segmentation_imgs = sorted(glob(f"{path}/tracks_0/*"))

        for img_path in tqdm(segmentation_imgs):
            img_name = f'{Path(img_path).stem}.tif'
            seg_img = io.imread(img_path)
            channel_1 = io.imread(f"{path}/background_corrected_1/{img_name}")
            channel_2 = io.imread(f"{path}/background_corrected_2/{img_name}")
            
            features = create_composite_features(feature_creator, channel_1, channel_2)
            
            neck_predictions = classifiers[strain]['neck_clf'].predict_segmenter(features)
            spindle_predictions = classifiers[strain]['spindle_clf'].predict_segmenter(features)
            
            io.imsave(f'{path}/active_neck_proba/{img_name}', neck_predictions[3].astype(np.float16))
            io.imsave(f'{path}/inactive_neck_proba/{img_name}', neck_predictions[4].astype(np.float16))
            io.imsave(f'{path}/spindle_proba/{img_name}', spindle_predictions[3].astype(np.float16))